In [4]:
from pathlib import Path
from gitsource import chunk_documents
from minsearch import Index

# 1. Read the books and remove blank lines
books = []

for file_path in sorted(Path("books_text").glob("*.md")):
    text = file_path.read_text(encoding="utf-8")

    books.append({
        "source": file_path.name,
        "content": [
            line
            for line in text.splitlines()
            if line.strip()
        ],
    })

if not books:
    raise FileNotFoundError(
        "No Markdown files found in books_text/. "
        "Check your notebook's working directory."
    )

# 2. Create chunks
chunks = chunk_documents(books, size=100, step=50)

# 3. Define the function before calling it
def prepare_documents(chunks):
    documents = []

    for chunk in chunks:
        document = chunk.copy()
        document["content"] = "\n".join(chunk["content"])
        documents.append(document)

    return documents

# 4. Prepare and index the chunks
documents = prepare_documents(chunks)

index = Index(text_fields=["content"])
index.fit(documents)

print("Number of documents indexed:", len(documents))

results = index.search(
    "python function definition",
    num_results=5,
)

if results:
    print("Top result came from:", results[0]["source"])
    print("\nContent preview:")
    print(results[0]["content"][:1000])
else:
    print("No results found.")


Number of documents indexed: 988
Top result came from: PhysicalModelingInMatlab4.md

Content preview:
| ------ | ------------ | --- | --- |
| total  | = total +    | a;  |     |
end
ans = total
Ifyouwereusinganyofthosevariablenamesbeforecallingthisscript, youmightbesurprised
to find, after running the script, that their values had changed. If you have two scripts that
48 Functions
use the same variable names, you might find that they work separately and then break when
you try to combine them. This kind of interaction is called a collision.
name
As the number of scripts you write increases, and they get longer and more complex, name
collisions become more of a problem. Avoiding this problem is one of several motivations for
functions.
| 5.2 | Defining | Functions |     |     |
| --- | -------- | --------- | --- | --- |
A function is like a script, except that each function has its own workspace, so any variables
defined inside a function only exist while the function is running and don

In [5]:
from openai import OpenAI

openai_client = OpenAI()
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context,
    ).strip()
    return prompt


def search(question):
    return index.search(question, num_results=5)


def llm(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages,
    )

    return {
        "answer": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [6]:
result = rag("python function definition")

print("Response:\n")
print(result["answer"])

print("\nInput tokens:", result["input_tokens"])
print("Output tokens:", result["output_tokens"])

Response:

In Python, a function is defined using the `def` keyword, followed by the function name and parentheses that may include parameters. Here's a basic structure of a function definition:

```python
def function_name(parameters):
    # Code block
    return output_value
```

### Example
Here's an example of a simple function that adds two numbers:

```python
def add_numbers(a, b):
    result = a + b
    return result
```

### Key Points
- `def`: This keyword indicates that you are defining a function.
- `function_name`: This is the name you choose for your function, and it should be descriptive of its purpose.
- `parameters`: These are optional inputs that you can pass into the function; they allow you to pass data to the function.
- `return`: This statement allows the function to send back a value or result to the caller.

After defining a function, you can call it like this:

```python
sum_result = add_numbers(3, 5)
print(sum_result)  # Output: 8
```

This is a basic overview 

In [7]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(
        description="The main answer to the user's question in markdown"
    )
    found_answer: bool = Field(
        description="True if relevant information was found in the documentation"
    )
    confidence: float = Field(
        description="Confidence score from 0.0 to 1.0"
    )
    confidence_explanation: str = Field(
        description="Explanation about the confidence level"
    )
    answer_type: Literal[
        "how-to",
        "explanation",
        "troubleshooting",
        "comparison",
        "reference",
    ] = Field(description="The category of the answer")

    followup_questions: list[str] = Field(
        description="Suggested follow-up questions"
    )

In [8]:
def llm(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse,
    )

    return {
        "answer": response.output_parsed,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [9]:
structured_result = rag("python function definition")

unstructured_input_tokens = 7926
structured_input_tokens = structured_result["input_tokens"]

difference = structured_input_tokens - unstructured_input_tokens

event = structured_result["answer"]

if event is not None:
    print("Answer:\n")
    print(event.answer)

print("\nQ5 input tokens:", unstructured_input_tokens)
print("Structured input tokens:", structured_input_tokens)
print("MORE input tokens:", difference)

Answer:

### Python Function Definition

In Python, a function is defined using the `def` keyword, followed by the function name and parentheses which can include parameters. Here's a simple structure:

```python
def function_name(parameters):
    # Body of the function
    # Perform operations
    return result
```

#### Example of a Function Definition
Here’s a basic example of a function that adds two numbers:

```python
def add_numbers(a, b):
    result = a + b
    return result
```

In this example:
- `add_numbers` is the name of the function.
- `a` and `b` are parameters (inputs) to the function.
- The function computes the sum of `a` and `b` and returns it.

#### Calling a Function
You can call the function by using its name followed by arguments in parentheses:

```python
sum = add_numbers(3, 5)
print(sum)  # This will output: 8
```

### Key Points
- **Indentation**: Python uses indentation to define the blocks of code. The body of the function must be indented.
- **Return Stat